In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
headers = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/47.0.2526.106 Safari/537.36'}

main_page_1 = 'https://www.transfermarkt.com/wettbewerbe/'
main_page_2 = '/wettbewerbe?ajax=yw1&plus=1&page='
lands = ['europa', 'amerika', 'afrika', 'asien']
lands_while_end = {'europa': 'Third Tier',
                   'amerika': 'Third Tier',
                   'afrika': 'Domestic Cup',
                   'asien': 'Third Tier'}

In [20]:
# deklaracja tablic do stworzenia data setu o zawodnikach
league_names = []
leagues_tier = []
countries = []
clubs_nos = []
players_nos = []
foreigners_percentages = []
gpms = []
total_values = []

In [6]:
def get_value(value):
    int_value = value[1:-1]
    multipler = value [-1]
    if multipler == 'n':
        return int(float(value[1:-2])*1000000000)
    elif multipler == 'm':
        return int(float(int_value)*1000000)
    else:
        return int(float(int_value)*1000)

In [21]:
for continent in lands:
    loop_end = lands_while_end[continent]
    actual_tier = ""
    for page_number in range (1,7):
        url = main_page_1 +  continent + main_page_2 + str(page_number)
        pageTree = requests.get(url, headers=headers)
        pageSoup = BeautifulSoup(pageTree.content, 'lxml')
        leagues = pageSoup.find("table", {"class": "items"}).find("tbody").find_all("tr")
        for league in leagues:
            league_data = league.find_all("td")

            if len(league_data) == 1:
                actual_tier = league_data[0].contents[0].strip()

            elif len(league_data)> 2:
                league_name = league_data[2].find("a").contents[0]
                country = league_data[3].find("img")['title']
                clubs_no = league_data[4].contents[0]
                players_no = league_data[5].contents[0]
                f_p = league_data[7].find("a").contents[0]
                if continent == 'europa':
                    gpm = league_data[9].find("a").contents[0]
                else:
                    gpm = league_data[8].find("a").contents[0] if league_data[8].find("a") else None
                
                value = league_data[-1].contents[0]

                league_names.append(league_name)
                leagues_tier.append(actual_tier)
                countries.append(country)
                clubs_nos.append(clubs_no)
                players_nos.append(players_no)
                foreigners_percentages.append(round(float(f_p[:-2])/100*float(players_no)))
                gpms.append(gpm)
                total_values.append(get_value(value) if value != '-' else None)

            if actual_tier == loop_end:
                break
        
        if actual_tier == loop_end:
            break


In [55]:
# budujemy data frame'a
df = pd.DataFrame({
    "League_name":league_names,
    "League_tier":leagues_tier,
    "Country":countries,
    "Club_no":clubs_nos, 
    "Players_no":players_nos, 
    "Foreigners": foreigners_percentages, 
    "Goals_per_match": gpms, 
    "Total_value": total_values, 
    })

In [57]:
df

,League_name,League_tier,Country,Club_no,Players_no,Foreigners,Goals_per_match,Total_value
0,Premier League,First Tier,England,20,548,388,2.82,1.253000e+10
1,LaLiga,First Tier,Spain,20,500,216,2.57,5.570000e+09
2,Serie A,First Tier,Italy,20,572,383,2.36,5.430000e+09
3,Bundesliga,First Tier,Germany,18,520,304,3.22,4.660000e+09
4,Ligue 1,First Tier,France,18,478,299,2.84,3.640000e+09
...,...,...,...,...,...,...,...,...
213,NPL New South Wales,Second Tier,Australia,16,569,73,2.98,2.400000e+06
214,NPL Victoria,Second Tier,Australia,14,440,77,3.30,1.200000e+06
215,Premier League 2,Second Tier,Singapore,8,142,13,3.74,1.850000e+05
216,Hong Kong First Division,Second Tier,Hongkong,14,376,40,3.94,1.000000e+04


In [58]:
df.to_csv('leagues.csv', index=True)  